# Lesson 8 · Evaluation and validation

This is where the contest is won or lost. A model that looks good on the public leaderboard can be weak on the hidden one, which is what counts at the end. This module is about having justified trust in your score. It's the least flashy part and the one that makes the difference between a top 10 and a mid-table finish.

*The cells below are the code from the lesson, in order. Some of them expect data you load yourself (`df`, `X_train`, `y_train`).*

## 01 · Why accuracy lies to you

Accuracy is the share of correct predictions. It sounds reasonable, but on imbalanced classes it's misleading. If 98% of the examples are the no class, a model that always says no gets 98% accuracy and zero value. That's why you need metrics that look at each class separately.

## 02 · The confusion matrix: the foundation

The confusion matrix counts the four kinds of outcome for a binary classification. It's the basis every other metric is computed from, so it's worth understanding well.

- TP (true positive): it was positive, you said positive. Correct.
- TN (true negative): it was negative, you said negative. Correct.
- FP (false positive): it was negative, you said positive. A false alarm.
- FN (false negative): it was positive, you said negative. You missed it.

Which one hurts more depends on the problem. For a disease test, a false negative (a sick person sent home) is worse than a false positive. For a spam filter, it's the other way around. The right metric reflects the real cost of each kind of mistake.

## 03 · Precision, recall, F1

**precision = TP / (TP + FP)**

Of all the ones you called positive, how many really were. High precision = few false alarms.

**recall = TP / (TP + FN)**

Of all the ones that really were positive, how many you caught. High recall = you miss few.

Precision and recall pull in opposite directions. If you say positive only when you're very sure, precision goes up but recall goes down. If you say positive often, it's the reverse. F1 reconciles them into a single number, their harmonic mean, which is small if either of them is small.

**F1 = 2 · (precision · recall) / (precision + recall)**

The harmonic mean. It punishes imbalance: you can't have a high F1 with a tiny recall.

> **Remember:** With many classes, F1 is aggregated. Macro-F1 averages over classes treating them equally (good when you care about the rare ones). Weighted-F1 accounts for each class's size. Read in the rules which one is scored and optimize exactly that.

## 04 · Validation without leaks

Data leakage is when, without meaning to, information from the test set makes it into training. The result: a great local score that collapses on the real leaderboard. It's the sneakiest way to lose points, because everything looks fine.

You defend against it by splitting the data correctly. Keep a validation set the model doesn't see during training and use it to estimate the real score. Better, use k-fold: you split the data into k parts, train on k-1 and test on one, rotating, then average. That way you use all the data and get a more stable estimate.

When the classes are imbalanced, use stratified k-fold, which keeps the class proportions in each fold. Otherwise a fold might not contain the rare class at all. When the data has time or groups (the same patient in several rows), the split has to respect them, otherwise the model peeks.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, X, y, cv=cv, scoring="f1_macro")
print(scores.mean(), scores.std())

## 05 · Bias, variance and the learning curve

Two opposite diseases. High bias (underfitting): the model is too simple, it does badly on both training and validation. High variance (overfitting): the model is too complex, it does great on training but badly on validation, because it memorized.

The learning curve diagnoses them: you plot the training score and the validation score as you add more data. If both are low and close together, you have bias, you need a stronger model. If there's a big gap between them, you have variance, you need more data or regularization.

## 06 · The two final submissions

> **Remember:** The golden rule: at the end, choose your two submissions on purpose. One on your best local score (the validation), one on your best public leaderboard. If local and public agree, trust it. If they differ a lot, you have a leak or a bad split. Don't leave your final submissions to be whatever you sent last out of reflex.